# Agentic Fleet Orchestration with Strands Agents

Give **one agent one goal in plain English**, and watch it plan across a fleet of
robots, assign each sub-task to a robot that can do it, and dispatch over a safe
command mesh, from simulation to hardware.

This is the companion notebook to the blog post *Agentic AI for autonomous fleet
operations*. It is the notebook form of
[`fleet_orchestration_demo.py`](https://github.com/strands-labs/robots), and every
cell runs the real Strands Robots API.

**What you will do:**

1. Bring up a heterogeneous fleet in one simulated world and read each robot's capabilities.
2. Give a natural-language goal and watch a planner decompose it into sub-tasks.
3. Dispatch the assignments in a synchronized multi-robot control loop.
4. Take a robot offline mid-mission and see the plan re-form safely.
5. (Optional) Dispatch over a Zenoh peer mesh with a human-in-the-loop approval on every action.

**The whole default path runs on a laptop, CPU only. No GPU, no cloud credentials.**

## Setup

Install Strands Robots with the simulation extra. The `mesh` extra is only needed
for the optional Step 5.

```bash
uv pip install -U "strands-robots[sim-mujoco,mesh]"
```

On macOS set the MuJoCo GL backend to `cgl`; on headless Linux use `egl`. This must
be set before `strands_robots` triggers the first `import mujoco`.

In [ ]:
import os
import sys

# MuJoCo locks its GL backend at first import, so set this before importing the SDK.
# cgl is macOS-only; egl is the headless-Linux backend. An exported MUJOCO_GL wins.
os.environ.setdefault("MUJOCO_GL", "cgl" if sys.platform == "darwin" else "egl")

import logging
logging.basicConfig(level=logging.INFO, format="%(message)s")

from strands_robots.simulation import Simulation
from strands_robots.policies.mock import MockPolicy
print("Strands Robots ready.")

## Step 1 - Bring up a fleet and read its capabilities

A fleet is a set of robots with different physical abilities. What the allocator
needs to know is **what each robot can do**, not what it is: an arm offers
`manipulate`, a mobile base offers `navigate` and `transport`.

The registry records a `category` for every robot it knows (`arm`, `mobile`,
`mobile_manip`, `humanoid`, ...) but no capability field, so the mapping from
category to capabilities is orchestration policy and is defined below. Deriving it
from `category` rather than listing robots by name means a new vendor's robot is
classified as soon as the registry knows it, and an unmapped category raises
instead of silently defaulting to something the robot cannot do. The allocator
then matches a sub-task's required capability against that set, so adding a
capability class is one row in a map, not new planner code.

**`grasp` is the exception, and it is the interesting one.** A category cannot
answer it: 23 registry robots have category `arm` and 3 of them declare a `gripper`
block. So `grasp` is claimed from that block alone - the field that names the
gripper's actuators and which end of their range is closed, i.e. the only field
that says the mechanism exists. Over-advertising a capability is the one
allocation error an allocator cannot detect afterwards: the robot accepts the
task, the plan looks complete, and the failure happens in the world.


In [ ]:
# Capabilities are derived from the registry's own fields rather than hard-coded per
# robot, so a robot the registry knows about is classified without editing this
# notebook. The registry has no capability field of its own; the mapping below is
# the orchestration policy, and it lives here.
#
# `grasp` is deliberately NOT claimed from the category. It is claimed only when the
# registry entry carries a `gripper` block, because that block is the only thing that
# says the mechanism exists - it names the actuators and which end of their range is
# closed. Measured on the shipped registry, 23 robots have category `arm` and 3 of
# them declare a gripper, so inferring `grasp` from the category would advertise it
# for 20 arms that never declared one. That is the one allocation error an allocator
# cannot detect afterwards: the robot accepts the task and fails in the world. The
# library resolves a gripper the same way and for the same reason - see
# `MuJoCoMotionPrimitives._registry_gripper_metadata`, which prefers this block over
# its name heuristic rather than the reverse.
from strands_robots.registry import get_robot, resolve_name

CATEGORY_CAPABILITIES = {
    "arm":          {"manipulate"},
    "bimanual":     {"manipulate"},
    "hand":         {"manipulate"},
    "mobile":       {"navigate", "transport"},
    "mobile_manip": {"navigate", "transport", "manipulate"},
    "humanoid":     {"navigate", "manipulate"},
    "aerial":       {"navigate", "inspect"},
    # Mapped to nothing on purpose: an expressive robot (`reachy_mini`) has no
    # capability this allocator dispatches, so every sub-task it is offered is left
    # UNASSIGNED. Empty and explicit, rather than unmapped and raising, because
    # "advertises nothing" is a real answer and a missing category is a gap.
    "expressive":   set(),
}


def capabilities_of(robot_type):
    """Capability tags for a registry name or alias.

    Raises `ValueError` for a name the registry does not know and for a category
    this notebook has no row for. Neither case may return a default: a robot that
    quietly advertises `manipulate` gets handed work it may be unable to do.
    """
    entry = get_robot(robot_type)
    if entry is None:
        raise ValueError(
            f"{robot_type!r} is not in the robot registry (aliases resolve too, so "
            "check the spelling against `list_robots()`). Capabilities cannot be "
            "guessed for an unknown robot."
        )
    category = entry.get("category")
    caps = CATEGORY_CAPABILITIES.get(category)
    if caps is None:
        raise ValueError(
            f"{robot_type!r} (canonical {resolve_name(robot_type)!r}) has category "
            f"{category!r}, which this notebook maps to no capabilities. Add it to "
            "CATEGORY_CAPABILITIES before dispatching to it."
        )
    caps = set(caps)
    if entry.get("gripper"):
        caps.add("grasp")
    return caps


for name in ("so101", "ur5e", "go2", "reachy_mini"):
    print(f"{name:12s} {sorted(capabilities_of(name))}")


### The fleet itself

With the capability model in place, bring up the world. `arm_a` and `arm_b` declare
grippers, so they advertise `grasp`; `go2` resolves through its alias to
`unitree_go2` (category `mobile`) and advertises `navigate` and `transport`.


In [ ]:
# Stand up a heterogeneous fleet in one world: two arms + one mobile base.
sim = Simulation()
sim.create_world()

FLEET_SPEC = [
    ("arm_a", "so101", [0.0, 0.0, 0.0]),
    ("arm_b", "so100", [0.6, 0.0, 0.0]),
    ("base",  "go2",   [0.0, 0.8, 0.0]),
]

fleet = {}  # instance name -> {type, capabilities, online}
for name, rtype, pos in FLEET_SPEC:
    res = sim.add_robot(name=name, data_config=rtype, position=pos)
    assert res.get("status") == "success", res
    fleet[name] = {"type": rtype, "capabilities": capabilities_of(rtype), "online": True}

for name, m in fleet.items():
    print(f"{name:6s} [{m['type']:6s}] {sorted(m['capabilities'])}")


## Step 2 - Give a goal, and watch it plan

The planner turns one natural-language goal into a set of sub-tasks, each tagged
with the capability it needs, then assigns each to a capable online robot.

We show the **rule-based planner** first: it needs no model provider, so the whole
pipeline is provable end-to-end before an LLM is introduced. The
*allocation* (the safety-relevant "which robot runs this" decision) stays
deterministic and auditable no matter which planner produces the sub-tasks. Step 2b
swaps in a Strands agent that produces the same sub-task shape by reasoning instead
of keyword matching.

In [ ]:
def decompose_rule(goal):
    """Keyword decomposition: the deterministic floor, no model needed."""
    g = goal.lower()
    tasks = []
    if any(w in g for w in ("move", "transport", "bring", "carry", "fetch")):
        tasks.append({"description": "transport the bin to the work zone", "capability": "transport"})
    if any(w in g for w in ("sort", "pick", "place", "grasp", "load")):
        tasks.append({"description": "pick and place the parts", "capability": "grasp"})
    if not tasks:
        tasks.append({"description": goal.strip(), "capability": "manipulate"})
    return tasks

def allocate(tasks, fleet):
    """Assign each sub-task to a capable ONLINE robot, load-balanced.

    A robot may hold more than one sub-task, so we do not claim it after the
    first. Among capable online robots pick the least-loaded, ties by fleet order.
    An unmatched task is left UNASSIGNED - the safe outcome, never mis-assigned.
    """
    load = {name: 0 for name in fleet}
    order = list(fleet)
    for t in tasks:
        candidates = [n for n, m in fleet.items() if m["online"] and t["capability"] in m["capabilities"]]
        if not candidates:
            t["assigned_to"] = None
            continue
        chosen = min(candidates, key=lambda n: (load[n], order.index(n)))
        t["assigned_to"] = chosen
        load[chosen] += 1
    return tasks

def render_plan(goal, tasks):
    assigned = sum(t["assigned_to"] is not None for t in tasks)
    print(f'Goal: "{goal}"')
    print(f"Plan ({assigned}/{len(tasks)} assigned):")
    for i, t in enumerate(tasks, 1):
        who = t["assigned_to"] or "UNASSIGNED (no capable online robot)"
        print(f"  {i}. [{t['capability']}] {t['description']} -> {who}")

GOAL = "Move the parts bin to the work zone and sort the parts into trays."
tasks = allocate(decompose_rule(GOAL), fleet)
render_plan(GOAL, tasks)

### Step 2b (optional) - Let a Strands agent do the decomposition

Same sub-task shape, produced by reasoning instead of keyword matching. This cell
needs a Strands model provider (for example Amazon Bedrock credentials). If none is
available it falls back to the rule-based planner, so the notebook still runs.

Only the *decomposition* is delegated to the model; `allocate(...)` is unchanged, so
the assignment stays deterministic.

In [ ]:
import json

def decompose_agent(goal, fleet, model_id=None, region=None):
    from strands import Agent
    from strands.models import BedrockModel

    system = (
        "You decompose a robot-fleet goal into sub-tasks. Respond with ONLY a JSON "
        "array. Each element has 'description' (short imperative) and 'capability' "
        "(one of: manipulate, grasp, navigate, transport, inspect). Produce the "
        "minimum sub-tasks that cover the goal. No prose."
    )
    model = BedrockModel(model_id=model_id or "us.anthropic.claude-opus-4-8",
                         region_name=region or os.getenv("AWS_REGION", "us-east-1"))
    caps = sorted({c for m in fleet.values() for c in m['capabilities']})
    # callback_handler=None silences the SDK's default stdout streaming.
    agent = Agent(model=model, system_prompt=system, callback_handler=None)
    raw = str(agent(f'Fleet capabilities: {caps}. Goal: "{goal}". Return the JSON array.')).strip()
    s, e = raw.find("["), raw.rfind("]")
    data = json.loads(raw[s:e + 1]) if s != -1 and e != -1 else []
    out = [{"description": str(d.get("description", "task")), "capability": str(d["capability"])}
           for d in data if isinstance(d, dict) and d.get("capability")]
    if not out:
        raise ValueError(f"agent returned no parseable sub-tasks: {raw[:200]!r}")
    return out

try:
    agent_tasks = allocate(decompose_agent(GOAL, fleet), fleet)
    print("Agent planner:\n")
    render_plan(GOAL, agent_tasks)
except Exception as e:
    print(f"Agent planner unavailable ({type(e).__name__}: {e});\nfalling back to rule-based plan above.")

## Step 3 - Dispatch in a synchronized control loop

With assignments in hand, drive every assigned robot together in one control loop.
A robot that owns several sub-tasks runs one policy with its instructions joined, so
`run_multi_policy` gets exactly one entry per robot.

Here each robot runs a `MockPolicy` so the loop runs anywhere with no model weights.
On hardware you would swap in a real policy per robot; the dispatch code does not change.

In [ ]:
per_robot = {}
for t in tasks:
    if t["assigned_to"]:
        per_robot.setdefault(t["assigned_to"], []).append(t["description"])

policies = {name: MockPolicy() for name in per_robot}
instructions = {name: "; then ".join(descs) for name, descs in per_robot.items()}

result = sim.run_multi_policy(
    policies=policies,
    instructions=instructions,
    n_steps=30,
    control_frequency=50.0,
    action_horizon=4,
)
print(result.get("content", [{}])[0].get("text", result))

## Step 4 - Handle failure without mis-assigning work

A fleet plan is only credible if it degrades safely. Take the mobile base offline and
re-plan. No other online robot advertises `transport`, so that sub-task is left
**unassigned** rather than handed to a grasping arm that cannot carry a bin. The
grasp sub-task still routes to an arm.

Leaving it unassigned is the correct outcome: silently reassigning it would produce a
confident failure.

In [ ]:
fleet["base"]["online"] = False
print("base is now OFFLINE. Re-planning...\n")

tasks_after = allocate(decompose_rule(GOAL), fleet)
render_plan(GOAL, tasks_after)

## Step 5 (optional) - Dispatch over the mesh, with a human in the loop

The steps above dispatched in-process. In a real deployment the robots are separate
peers on a [Zenoh](https://zenoh.io/) mesh, and the orchestrator sends each
assignment as a mesh command. **Every physically-actuating action pauses for a human
approval**, delivered out-of-band of the model's tool arguments, so a prompt-injection
attempt cannot smuggle its own approval flag into the command body.

This cell needs the `mesh` extra and stands up a second peer, so it is heavier than
the default path. The approval function here auto-approves for an unattended run; in
an operator setting you would prompt a human. A declined action is never dispatched
and, per the mesh's rate-limit semantics, consumes no rate-limit slot.

> This cell is illustrative of the mesh API shape. Run it only with the `mesh` extra
> installed; it is safe to skip and does not affect Steps 1-4.

In [ ]:
def auto_approve(action, target, instruction):
    """Out-of-band approver. Returns a bool the model cannot influence."""
    print(f"[HITL] auto-approved {action} -> {target}: {instruction!r}")
    return True

def dispatch_over_mesh(sim, orch_mesh, per_robot, approve=auto_approve, n_steps=30):
    sim_peer = sim.peer_id
    dispatched, denied = [], []
    for name, descs in per_robot.items():
        instruction = "; then ".join(descs)
        # HITL gate BEFORE the command goes on the wire.
        if not approve("start", name, instruction):
            denied.append(name)
            continue
        r = orch_mesh.send(
            sim_peer,
            {"action": "start", "instruction": instruction,
             "policy_provider": "mock", "robot_name": name, "n_steps": n_steps},
            timeout=15.0,
        )
        ok = isinstance(r, dict) and r.get("result", {}).get("status") == "success"
        dispatched.append((name, ok))
    return {"dispatched": dispatched, "denied": denied}

# To run this end to end, build the fleet with the mesh joined:
#
#   os.environ.setdefault("STRANDS_MESH_LOCAL_DEV", "1")  # dev-only: no mTLS
#   from strands_robots.mesh import init_mesh
#   sim_mesh = init_mesh(sim, peer_id=None, peer_type="sim")
#   sim.mesh, sim.peer_id = sim_mesh, sim_mesh.peer_id
#   ... add robots ...
#   orch_mesh = init_mesh(_Owner(), peer_id="fleet-orchestrator")  # a SEPARATE peer
#   result = dispatch_over_mesh(sim, orch_mesh, per_robot)
#
# A peer cannot RPC its own peer_id in-process, so the orchestrator must be a
# distinct peer from the sim that hosts the robots.
print("Mesh dispatch helper defined. See the commented setup to run it with the mesh extra.")

## Clean up

The `Simulation` owns a thread pool, a MuJoCo world, and a temp directory. Always
destroy it when done.

In [ ]:
sim.destroy()
print("Fleet torn down.")

## Where to go from here

- **Hardware.** The `Robot()` abstraction is the same on real hardware: a physical
  SO-101 joins the same mesh, answers the same commands, and takes the same
  assignments. Natural-language control of a physical arm through this stack works
  today.
- **Real policies.** Swap `MockPolicy` for a policy trained on your robot and scene.
  Completing a manipulation task autonomously needs a fine-tune on teleoperated
  demonstrations - the record-train-deploy loop covered in the streaming data loop post.
- **Deploy the planner.** The planner is an agent that reasons and dispatches; it
  holds no model weights, runs on CPU, and deploys to Amazon Bedrock AgentCore the
  way any Strands agent does. Heavy vision-language-action inference runs on a
  separate GPU, called over the network.

See the [Strands Robots documentation](https://strands-labs.github.io/robots/) for
the robot catalog, simulation, policy providers, and the mesh in depth.